In [ ]:
!pip install -qU datasets pandas

In [ ]:
import pandas as pd
from datasets import load_dataset

REPO = "bakrianoo/jabarti-llm-dataset"

pretrain = load_dataset(
    REPO, "pretrain", split="phase1_train"
).to_pandas()

finetune = load_dataset(
    REPO, "finetune", split="train"
).to_pandas()

In [ ]:
print("=== pretrain schema ===")
print(pretrain.dtypes)

print("\n=== finetune schema ===")
print(finetune.dtypes)

=== pretrain schema ===
egypt_relevance         str
chunk_total           int64
topic_domain            str
importance              str
section_title           str
is_person              bool
chunk_strategy          str
ortho_convention        str
content_quality         str
pageviews_30d         int64
chunk_index           int64
language                str
source_type             str
dataset_version         str
is_stub_like           bool
pair_id                 str
article_id              str
title                   str
sample_weight       float64
category                str
text                    str
url                     str
dtype: object

=== finetune schema ===
difficulty           str
type                 str
qa_id                str
dataset_version      str
question             str
article_title        str
answer               str
source_article_id    str
language             str
dtype: object


In [ ]:
splits = {
    "pretrain / phase1_train": ("pretrain", "phase1_train"),
    "pretrain / phase1_eval":  ("pretrain", "phase1_eval"),
    "pretrain / phase2_train": ("pretrain", "phase2_train"),
    "pretrain / phase2_eval":  ("pretrain", "phase2_eval"),

    "finetune / train":        ("finetune", "train"),
    "finetune / eval":         ("finetune", "eval"),
}

# Rough token estimate for bilingual AR/EN with a BPE tokenizer at 24K vocab
CHARS_PER_TOKEN = 3.5

# Candidate model context lengths
CONTEXT_LENGTHS = [512, 1024, 2048]



for label_name, (subset, split_name) in splits.items():

    df = load_dataset(
        REPO, subset, split=split_name
    ).to_pandas()

    print(f"\n====== {label_name} ======")
    print("ROWS: ",df.shape[0])

    # =========================================================
    # NULL values
    # =========================================================

    nulls = df.isnull().sum()
    non_nulls = nulls[ nulls > 0 ]

    print(
        "NONE"
        if non_nulls.empty
        else non_nulls.to_string()
    )

    # =========================================================
    # Language distribution
    # =========================================================

    total_rows = df.shape[0]

    lang_counts = df["language"].value_counts()
    for lang, n in lang_counts.items():
        print(
            f"{lang} => {n} samples -- ",
            f"{n/total_rows*100:.1f}%"
        )

    # =========================================================
    # Character length distribution
    # =========================================================

    text_len_dist = 0
    if subset == "pretrain":
        text_len_dist = df["text"].str.len().describe().round(1)

    elif subset == "finetune":
        text_len_dist = pd.concat([
            df["question"].str.len(),
            df["answer"].str.len()
        ], ignore_index=True).describe().round(1)

    print("\n=== TEXT LENGTH DISTRIBUTION (characters) ===")
    print(text_len_dist)

    # =========================================================
    # Approximate token length per MODEL SAMPLE
    # =========================================================

    if subset == "pretrain":
        sample_chars = df["text"].str.len()

    elif subset == "finetune":
        sample_chars = (
            df["question"].str.len()
            +
            df["answer"].str.len()
        )

    approx_tokens = (sample_chars / CHARS_PER_TOKEN).astype(int)

    print("\n=== APPROXIMATE TOKEN COUNT PER SAMPLE ===")
    print(
        approx_tokens
        .describe()
        .round(1)
    )

    # =========================================================
    # Context length coverage
    # =========================================================

    for ctx in CONTEXT_LENGTHS:
        dropped_pct = (
            (approx_tokens > ctx).mean() * 100
        )

        kept_pct = 100 - dropped_pct

        print(
            f"current max_length={ctx} | "
            f"kept: {kept_pct:.1f}% | "
            f"dropped: {dropped_pct:.1f}%"
        )


    print(f"====== ====== ======")



====== pretrain / phase1_train ======
ROWS:  175476
NONE
en => 125766 samples --  71.7%
ar => 49710 samples --  28.3%

=== TEXT LENGTH DISTRIBUTION (characters) ===
count    175476.0
mean       1553.0
std         702.1
min         178.0
25%        1169.0
50%        1653.0
75%        1913.0
max       20049.0
Name: text, dtype: float64

=== APPROXIMATE TOKEN COUNT PER SAMPLE ===
count    175476.0
mean        443.3
std         200.6
min          50.0
25%         334.0
50%         472.0
75%         546.0
max        5728.0
Name: text, dtype: float64
current max_length=512 | kept: 61.9% | dropped: 38.1%
current max_length=1024 | kept: 99.3% | dropped: 0.7%
current max_length=2048 | kept: 99.8% | dropped: 0.2%
====== ====== ======

====== pretrain / phase1_eval ======
ROWS:  2000
NONE
en => 1426 samples --  71.3%
ar => 574 samples --  28.7%

=== TEXT LENGTH DISTRIBUTION (characters) ===
count    2000.0
mean     1565.1
std       577.7
min       268.0
25%      1226.8
50%      1666.5
75%      1

In [ ]:
df = load_dataset(REPO, "pretrain", split="phase1_train").to_pandas()

text_dups = df.duplicated(
    subset=["text"],
).sum()

print(f"Rows with exact duplicate text: {text_dups}")

pair_dups = df.duplicated(
    subset=["article_id", "chunk_index"]
).sum()

print(f"Duplicate (article_id, chunk_index) pairs: {pair_dups}")

Rows with exact duplicate text: 0
Duplicate (article_id, chunk_index) pairs: 0
